# Control de Lectura 3 — Verificación numérica de gradientes mediante diferencias finitas

**Nombre:** _[escriba aquí su nombre]_ · **Función implementada:** `diferencia_central(f, w, h)` (método de diferencias centrales)

**Objetivo.** Contrastar dos gradientes candidatos con diferencias centrales e interpretar el efecto del paso y de la precisión numérica, sin derivar a mano ni entrenar.

**Contexto.** Se proporciona la función f(w) = w³ − 2w y dos candidatos a su derivada, gA(w) = 3w² − 2 y gB(w) = 3w − 2. La verificación se realiza en w = 2.0 con los pasos h = 1e-1, 1e-5 y 1e-16, usando float de Python y sin redondear los cálculos intermedios.

## Tareas

Esta sección cubre las tres tareas del control. La Tarea 1 (registrar la predicción) se resuelve a continuación en Markdown, antes de ejecutar cualquier cálculo. Las Tareas 2 y 3 (completar la función y ejecutar y observar) se resuelven en la celda de código siguiente, que define f, gA y gB, completa `diferencia_central` con el método central, imprime la evidencia legible de los tres casos (incluido el caso degenerado) y guarda `resultados_control_3.csv`.

### Tarea 1 — Predicción (antes de ejecutar)

**P0. ¿Qué dos cantidades se comparan al verificar un gradiente?** Se comparan, por un lado, el valor que entrega el candidato g(w) en el punto de evaluación w y, por otro, la aproximación numérica de f'(w) obtenida con la diferencia central (f(w+h) − f(w−h)) / (2h). El candidato es compatible con la evidencia si ambas cantidades coinciden dentro de una tolerancia razonable.

**¿Un h menor siempre mejora la aproximación?** Mi predicción es que **no**. Al reducir h disminuye el error de truncamiento del método (de orden O(h²) en la diferencia central), pero crece el efecto del redondeo de la aritmética de punto flotante: al restar f(w+h) − f(w−h), dos valores casi iguales, se produce cancelación catastrófica; y si h es tan pequeño que w+h o w−h se redondean de vuelta a w, la aproximación se degrada por completo. Por tanto espero que un paso intermedio pequeño dé la fila más fiable, y no el h más pequeño disponible. **Conservo esta predicción** aunque la evidencia luego la corrija.

In [1]:
# -*- coding: utf-8 -*-
"""
CONTROL DE LECTURA 3 - Verificación numérica de gradientes mediante diferencias finitas.
Subtarea s1: script único de evidencia numérica (sin gráficos).

Datos del problema (literales del enunciado):
    f(w)  = w**3 - 2*w
    gA(w) = 3*w**2 - 2      (candidato A a la derivada)
    gB(w) = 3*w - 2         (candidato B a la derivada)
    punto de evaluación w = 2.0
    pasos h = 1e-1, 1e-5, 1e-16 (en ese orden)

Salidas:
    * tabla legible en stdout con toda la evidencia de los tres casos
      (se conservan los tres, incluido el caso degenerado h = 1e-16);
    * resultados_control_3.csv (UTF-8, separador coma);
    * resultados.json (todos los números calculados, claves descriptivas, floats planos).

Aritmética: float de Python, sin redondear cálculos intermedios.
"""

import csv
import json

# ---------------------------------------------------------------------------
# 1) Función objetivo y candidatos (definiciones literales del enunciado)
# ---------------------------------------------------------------------------

def f(w):
    """Función objetivo f(w) = w**3 - 2*w."""
    return w**3 - 2*w


def gA(w):
    """Candidato A a la derivada: gA(w) = 3*w**2 - 2."""
    return 3*w**2 - 2


def gB(w):
    """Candidato B a la derivada: gB(w) = 3*w - 2."""
    return 3*w - 2


# ---------------------------------------------------------------------------
# 2) Función a completar: diferencia central.
#    SOLO el método central: (f(w + h) - f(w - h)) / (2 * h).
#    Dentro de esta función NO se usa la derivada analítica ni los candidatos.
# ---------------------------------------------------------------------------

def diferencia_central(f, w, h):
    """Aproximación de la derivada por diferencia central."""
    return (f(w + h) - f(w - h)) / (2 * h)


# ---------------------------------------------------------------------------
# 3) Ejecución de los tres casos (float de Python, sin redondeos)
# ---------------------------------------------------------------------------

W = 2.0
PASOS = [1e-1, 1e-5, 1e-16]

casos = []
for h in PASOS:
    aproximacion = diferencia_central(f, W, h)          # float, sin redondear
    valor_gA = gA(W)                                    # candidato A en w
    valor_gB = gB(W)                                    # candidato B en w
    disc_gA = abs(aproximacion - valor_gA)              # discrepancia absoluta vs A
    disc_gB = abs(aproximacion - valor_gB)              # discrepancia absoluta vs B
    w_mas_h = W + h                                     # punto perturbado superior
    w_menos_h = W - h                                   # punto perturbado inferior
    casos.append({
        "h": float(h),
        "aproximacion": float(aproximacion),
        "gA": float(valor_gA),
        "gB": float(valor_gB),
        "disc_gA": float(disc_gA),
        "disc_gB": float(disc_gB),
        "w_mas_h": float(w_mas_h),
        "w_menos_h": float(w_menos_h),
        "igualdad_w_mas_h": bool(w_mas_h == W),
        "igualdad_w_menos_h": bool(w_menos_h == W),
    })

# ---------------------------------------------------------------------------
# 4) Tabla legible en stdout (evidencia completa de los tres casos)
# ---------------------------------------------------------------------------

SEP = "=" * 79
print(SEP)
print("CONTROL DE LECTURA 3 - Verificación numérica de gradientes")
print("Método central: diferencia_central(f, w, h) = (f(w+h) - f(w-h)) / (2*h)")
print("f(w) = w**3 - 2*w   |   gA(w) = 3*w**2 - 2   |   gB(w) = 3*w - 2")
print("w = 2.0   |   h = 1e-1, 1e-5, 1e-16   |   float de Python, sin redondeos")
print(SEP)

for i, c in enumerate(casos, start=1):
    print()
    print("Caso %d de %d:  h = %r" % (i, len(casos), c["h"]))
    print("-" * 79)
    print("  aproximacion = diferencia_central(f, 2.0, h) : %r" % c["aproximacion"])
    print("  gA(2.0) = 3*(2.0)**2 - 2                     : %r" % c["gA"])
    print("  gB(2.0) = 3*(2.0) - 2                        : %r" % c["gB"])
    print("  disc_gA = abs(aproximacion - gA(2.0))        : %r" % c["disc_gA"])
    print("  disc_gB = abs(aproximacion - gB(2.0))        : %r" % c["disc_gB"])
    print("  punto perturbado w + h   (repr exacto)       : %r" % c["w_mas_h"])
    print("  punto perturbado w - h   (repr exacto)       : %r" % c["w_menos_h"])
    print("  igualdad real (w + h == w)                   : %r" % c["igualdad_w_mas_h"])
    print("  igualdad real (w - h == w)                   : %r" % c["igualdad_w_menos_h"])

print()
print(SEP)
print("Resumen compacto (los tres casos se conservan como evidencia):")
print("%-9s %-24s %-24s %-24s %-8s %-8s" % (
    "h", "aproximacion", "disc_gA", "disc_gB", "w+h==w", "w-h==w"))
for c in casos:
    print("%-9r %-24r %-24r %-24r %-8r %-8r" % (
        c["h"], c["aproximacion"], c["disc_gA"], c["disc_gB"],
        c["igualdad_w_mas_h"], c["igualdad_w_menos_h"]))
print(SEP)

# ---------------------------------------------------------------------------
# 5) Guardar resultados_control_3.csv (UTF-8, separador coma)
# ---------------------------------------------------------------------------

COLUMNAS = ["h", "aproximacion", "gA", "gB", "disc_gA", "disc_gB",
            "w_mas_h", "w_menos_h", "igualdad_w_mas_h", "igualdad_w_menos_h"]

with open("resultados_control_3.csv", "w", newline="", encoding="utf-8") as fh:
    escritor = csv.writer(fh, delimiter=",")
    escritor.writerow(COLUMNAS)
    for c in casos:
        escritor.writerow([c[col] for col in COLUMNAS])

print()
print("Archivo escrito: resultados_control_3.csv (UTF-8, separador coma)")

# ---------------------------------------------------------------------------
# 6) Guardar resultados.json (todos los números, floats planos, indent=2)
# ---------------------------------------------------------------------------

resultados = {
    "control": "Control de Lectura 3 - verificación numérica de gradientes",
    "funcion_f": "w**3 - 2*w",
    "candidato_gA": "3*w**2 - 2",
    "candidato_gB": "3*w - 2",
    "metodo": "diferencia central: (f(w+h) - f(w-h)) / (2*h)",
    "w": 2.0,
    "pasos_h": [c["h"] for c in casos],
    "casos": casos,
}

with open("resultados.json", "w", encoding="utf-8") as fh:
    json.dump(resultados, fh, indent=2, ensure_ascii=False)

print("Archivo escrito: resultados.json")


CONTROL DE LECTURA 3 - Verificación numérica de gradientes
Método central: diferencia_central(f, w, h) = (f(w+h) - f(w-h)) / (2*h)
f(w) = w**3 - 2*w   |   gA(w) = 3*w**2 - 2   |   gB(w) = 3*w - 2
w = 2.0   |   h = 1e-1, 1e-5, 1e-16   |   float de Python, sin redondeos

Caso 1 de 3:  h = 0.1
-------------------------------------------------------------------------------
  aproximacion = diferencia_central(f, 2.0, h) : 10.010000000000007
  gA(2.0) = 3*(2.0)**2 - 2                     : 10.0
  gB(2.0) = 3*(2.0) - 2                        : 4.0
  disc_gA = abs(aproximacion - gA(2.0))        : 0.010000000000006892
  disc_gB = abs(aproximacion - gB(2.0))        : 6.010000000000007
  punto perturbado w + h   (repr exacto)       : 2.1
  punto perturbado w - h   (repr exacto)       : 1.9
  igualdad real (w + h == w)                   : False
  igualdad real (w - h == w)                   : False

Caso 2 de 3:  h = 1e-05
---------------------------------------------------------------------------

## Preguntas de interpretación

**P0 — Predicción registrada.** La predicción se registró antes de ejecutar y se conserva íntegra en la sección Tareas, bajo la etiqueta "Predicción (antes de ejecutar)". Las dos cantidades comparadas son el valor del candidato g(w) en w y la aproximación numérica de f'(w) por diferencia central; y un h menor no siempre mejora la aproximación, como confirma el contraste que se detalla en P2.

**P1 — Candidato respaldado por una fila fiable.** La fila fiable es la del paso intermedio h = 1e-5. Como muestra la salida anterior, su aproximación es 10.000000000198739; la discrepancia absoluta frente a gA(2.0) = 10.0 es 1.9873880319210002e-10 (prácticamente cero), mientras que frente a gB(2.0) = 4.0 es 6.000000000198739, del orden de la propia separación entre ambos candidatos. Esta evidencia respalda de manera clara al candidato **gA(w) = 3w² − 2**. La fila de h = 1e-1 también apunta a gA (discrepancia 0.010000000000006892 frente a 6.010000000000007), pero con un error de truncamiento ya visible; la fila de h = 1e-16 es degenerada (aproximación 0.0) y no sirve para decidir.

**P2 — Menor h frente a h intermedio.** Con el paso intermedio h = 1e-5 la aproximación es 10.000000000198739 y los puntos perturbados sí se separan de w: w + h = 2.00001 y w − h = 1.99999, con igualdad real False en ambos casos. Con el menor h = 1e-16 la aproximación colapsa a 0.0 y ambos puntos perturbados valen exactamente 2.0: w + h == w y w − h == w son True, de modo que el numerador f(w+h) − f(w−h) se anula y la división entrega cero. La causa es que 1e-16 está por debajo del épsilon de máquina relativo a 2.0: al sumarle o restarle ese paso, el resultado se redondea de vuelta a 2.0 y la perturbación se absorbe sin dejar rastro. Esto contrasta con —y confirma— la predicción de P0: reducir h mejora solo hasta cierto punto; después el redondeo domina al truncamiento y la aproximación empeora hasta volverse inútil.

**P3 — ¿Coincidir en w = 2 demuestra corrección para cualquier entrada?** No. La coincidencia en un único punto solo es evidencia local: dos funciones distintas pueden coincidir en w = 2 y divergir en cualquier otro punto, de modo que un candidato incorrecto podría pasar la prueba por casualidad justo en ese punto. Como comprobación adicional, sin programarla, propongo repetir la misma verificación por diferencia central en otros puntos de evaluación — por ejemplo w = −1, 0, 1 y 3, o en varios puntos aleatorios — y exigir que las discrepancias frente al candidato se mantengan pequeñas en todos ellos; solo un acuerdo consistente en múltiples puntos respaldaría la corrección del gradiente para cualquier entrada.

## Entrega

**Control:** Control de Lectura 3 — Verificación numérica de gradientes mediante diferencias finitas. **Función implementada:** `diferencia_central(f, w, h)`, que devuelve la diferencia central (f(w+h) − f(w−h)) / (2h) sin usar la derivada analítica ni los candidatos.

**Se entrega:**

- Este notebook ejecutado, con la función `diferencia_central` completada y la evidencia de los tres casos en w = 2.0 con h = 1e-1, 1e-5 y 1e-16 (las salidas impresas y el resumen compacto se conservan íntegros).
- Las respuestas P0 a P3, con la predicción P0 registrada antes de ejecutar y conservada.
- El archivo `resultados_control_3.csv` (UTF-8, separador coma), generado por el código, con las columnas h, aproximacion, gA, gB, disc_gA, disc_gB, w_mas_h, w_menos_h, igualdad_w_mas_h e igualdad_w_menos_h.

**Estado de la entrega:** completa; no queda nada pendiente.